# FILE SYSTEMS: Btrfs

## Background on File Systems
A file system manages how data is stored and retrieved on a disk. It provides a structured way to organize files, prevent conflicts, and optimize performance. Traditional file systems like ext4 and XFS are widely used, but newer systems like Btrfs introduce advanced features such as snapshots, copy-on-write (CoW), and built-in RAID support.

## Introduction to Btrfs
Btrfs (B-tree File System) is a modern file system designed to address the limitations of older file systems. It supports high scalability, self-healing mechanisms, and advanced storage management features.

### Key Features of Btrfs
* Copy-on-Write (CoW): Ensures data integrity by writing new changes to a different location before updating metadata.
* Snapshots: Enables point-in-time copies of data for backup and recovery.
* Subvolumes: Logical partitions within a Btrfs volume that function like separate file systems.
* Compression: Built-in support for transparent data compression (zlib, lzo, zstd).
* RAID Support: Supports RAID 0, 1, 10, 5, and 6 at the file system level.
* Deduplication: Reduces duplicate data storage, saving space.
* Self-healing & Checksumming: Protects against data corruption using checksums.

### Learning Objectives
* Understand the structure and advantages of Btrfs.
* Learn how to create and mount a Btrfs file system.
* Demonstrate key features such as subvolumes, snapshots, compression,CoW, deduplication and RAID.

For this courselet we will require 2 Disks each of 1 GB (one will be used for most of the operations and one more will be required for demonstrating RAID).

### 0. Set command to enable verbose logging in Bash
set command will be used here to reprint the command being run in the output. This will help in keeping the shell output clear and map easily between the commands and their outputs.

https://man7.org/linux/man-pages/man1/set.1p.html

In [3]:
set -x

+ set -x


### 1. Identify Available Disks
List available disks and partitions:

In [4]:
lsblk

+ lsblk
NAME   MAJ:MIN RM  SIZE RO TYPE MOUNTPOINT
loop0    7:0    0 91.9M  1 loop /snap/lxd/29619
loop1    7:1    0   64M  1 loop /snap/core20/2318
loop2    7:2    0 38.8M  1 loop /snap/snapd/21759
vda    252:0    0   20G  0 disk 
├─vda1 252:1    0  550M  0 part /boot/efi
├─vda2 252:2    0    8M  0 part 
└─vda3 252:3    0 19.5G  0 part /
vdb    252:16   0    1G  0 disk 
vdc    252:32   0    1G  0 disk 


### 2. Partition the disk
Most commercial PCs come with a single partition on their hard drive, but keeping all the data, applications, and operating system on the same partition can be risky because, if something happens to the partition’s index file, the computer won’t be able to boot up off that drive–and we won’t be able to access the rest of your data.

Partitioning the hard drive essentially means the portions of that drive will be treated as separate entities by the computer. If you keep your system and apps on a partition separate from your data (documents, music, video, and the like), the data will be easier to back up (because your backup utility won’t bother to copy the system and apps, which you can reinstall from the discs or redownload from an online source).

Choosing /dev/vdb as /dev/vda contains an operating system and partitioning this can make the system unbootable. We partition the disk using sgdisk as it is a command-line GPT manipulator for Linux and Unix.

https://linux.die.net/man/8/sgdisk

In [5]:
sudo sgdisk -n 1:2MiB:500MiB /dev/vdb

+ sudo sgdisk -n 1:2MiB:500MiB /dev/vdb
Creating new GPT entries in memory.
The operation has completed successfully.


sgdisk -n 1:2MiB:500MiB /dev/vdb, creates a new partition, numbered 1, starting at 2MiB and ending at 500MiB on /dev/vdb.

### 3. Verifying partitioning by listing all current partitions on /dev/vdb
We can check the available disks using the command "sudo fdisk -l /dev/vdb"

fdisk is a dialog-driven program for the creation and manipulation of partition tables. It understands GPT, MBR, Sun, SGI, and BSD partition tables. Block devices can be divided into one or more logical disks called partitions. This division is recorded in the Partition table, usually found in sector 0 of the disk.

https://man7.org/linux/man-pages/man8/fdisk.8.html

In [6]:
sudo fdisk -l /dev/vdb

+ sudo fdisk -l /dev/vdb
Disk /dev/vdb: 1 GiB, 1073741824 bytes, 2097152 sectors
Units: sectors of 1 * 512 = 512 bytes
Sector size (logical/physical): 512 bytes / 512 bytes
I/O size (minimum/optimal): 512 bytes / 512 bytes
Disklabel type: gpt
Disk identifier: 56BC065D-4AB0-45E1-AFF8-CA061BD3A532

Device     Start     End Sectors  Size Type
/dev/vdb1   4096 1024000 1019905  498M Linux filesystem


### 4. Create a Btrfs File System
If the partition is to be used to hold files, it needs a filesystem to manage the file infrastructure (keeping track of file locations, providing a directory structure, holding metadata about the file characteristics, permissions, etc.). Formatting creates the filesystem inside the partition.

mkfs is used to build a Linux filesystem on a device, usually a hard disk partition. 'mkfs.btrfs' is used to create the btrfs filesystem on a single or multiple devices.

https://btrfs.readthedocs.io/en/latest/mkfs.btrfs.html

https://man7.org/linux/man-pages/man8/mkfs.btrfs.8.html

If the partition already has a Linux filesystem, it will ask if you wish to proceed with the command anyway.

To override the warning we use yes command in combination with a pipe | to send "yes" as input to the command.

| (pipe) is a powerful feature that connects the output of one command to the input of another. This helps in creating a chain of commands that process data sequentially.

In [7]:
sudo mkfs.btrfs -L mybtrfs /dev/vdb1

+ sudo mkfs.btrfs -L mybtrfs /dev/vdb1
btrfs-progs v5.4.1 
See http://btrfs.wiki.kernel.org for more information.

Label:              mybtrfs
UUID:               1fca2654-5e68-48b1-85e0-260bafb99ce7
Node size:          16384
Sector size:        4096
Filesystem size:    498.00MiB
Block group profiles:
  Data:             single            8.00MiB
  Metadata:         DUP              32.00MiB
  System:           DUP               8.00MiB
SSD detected:       no
Incompat features:  extref, skinny-metadata
Checksum:           crc32c
Number of devices:  1
Devices:
   ID        SIZE  PATH
    1   498.00MiB  /dev/vdb1



Verify the file system:

In [8]:
sudo btrfs filesystem show

+ sudo btrfs filesystem show
Label: 'mybtrfs'  uuid: 1fca2654-5e68-48b1-85e0-260bafb99ce7
	Total devices 1 FS bytes used 128.00KiB
	devid    1 size 498.00MiB used 88.00MiB path /dev/vdb1



### 5. Creating a mount point to interact with the disk
A mount point is a directory or file at which a new filesystem, directory, or file is made accessible. By using a mount point, data stored on different physical and logical volumes can be put on the same filesystem. This way, all the data needed on the system can be accessed from the root directory.

The mkdir command creates directories. This command can create multiple directories at once as well as set the permissions for the directories. It is important to note that the user executing this command must have written permission in the parent directory to create a directory.

https://man7.org/linux/man-pages/man1/mkdir.1.html

In [9]:
sudo mkdir -p /mnt/btrfs

+ sudo mkdir -p /mnt/btrfs


### 6. Mounting the partition at the mount point
The filesystem must be mounted before it can be accessed. Mounting a filesystem attaches it to the file hierarchy at some path (mount point) and makes it available to the system.

This can be accomplished using the mount command. We mount our partition containing the btrfs filesystem that we created earlier on the directory /mnt/btrfs that we created in the previous step.

https://man7.org/linux/man-pages/man8/mount.8.html

In [10]:
sudo mount -t btrfs /dev/vdb1 /mnt/btrfs

+ sudo mount -t btrfs /dev/vdb1 /mnt/btrfs


### 7. Verifying if the partition is mounted
The df command is used to display the free disc space of a specific file system.

https://man7.org/linux/man-pages/man1/df.1.html

In [11]:
df -hT

+ df -hT
Filesystem     Type      Size  Used Avail Use% Mounted on
udev           devtmpfs  941M     0  941M   0% /dev
tmpfs          tmpfs     198M  1.1M  197M   1% /run
/dev/vda3      ext4       19G  3.3G   15G  19% /
tmpfs          tmpfs     986M     0  986M   0% /dev/shm
tmpfs          tmpfs     5.0M     0  5.0M   0% /run/lock
tmpfs          tmpfs     986M     0  986M   0% /sys/fs/cgroup
/dev/loop0     squashfs   92M   92M     0 100% /snap/lxd/29619
/dev/loop1     squashfs   64M   64M     0 100% /snap/core20/2318
/dev/vda1      vfat      549M  176K  549M   1% /boot/efi
/dev/loop2     squashfs   39M   39M     0 100% /snap/snapd/21759
tmpfs          tmpfs     198M     0  198M   0% /run/user/1000
/dev/vdb1      btrfs     498M  3.5M  417M   1% /mnt/btrfs


# Features of Btrfs file system

## Compression
Btrfs supports inline (on-the-fly) compression, which can:
* Reduce storage usage.
* Improve disk performance (especially on SSDs).
* Reduce write amplification on flash storage.

Btrfs supports three compression algorithms:
* zlib: Moderate compression, good balance between speed and space-saving.
* lzo: faster compression and decompression than zlib but a lower compression ratio.
* zstd: Best trade-off between compression ratio and speed.

### 1. Enabling Compression in Btrfs
You can enable compression while mounting a Btrfs partition using the **compress** option.

**Using LZO (Faster but Lower Compression)**

-o compress=lzo: Enables lzo compression.

https://btrfs.readthedocs.io/en/latest/Compression.html

In [12]:
sudo mount -o remount,compress=lzo /mnt/btrfs

+ sudo mount -o remount,compress=lzo /mnt/btrfs


### 2. Demonstrating Compression
We can test how Btrfs compresses newly written files by creating test files before and after enabling compression. For this first we disable compression temporarily by remounting. 

In [13]:
sudo mount -o remount,compress=no /mnt/btrfs

+ sudo mount -o remount,compress=no /mnt/btrfs


Now we create a large test file with the following command:
* dd: Copies raw data.
* if=/dev/zero: Reads from a source of zeroes.
* of=/mnt/btrfs/testfile: Writes output to testfile.
* bs=1M count=100: Creates a 100MB file.

In [14]:
sudo dd if=/dev/zero of=/mnt/btrfs/testfile bs=1M count=100

+ sudo dd if=/dev/zero of=/mnt/btrfs/testfile bs=1M count=100
100+0 records in
100+0 records out
104857600 bytes (105 MB, 100 MiB) copied, 0.0642597 s, 1.6 GB/s


Now, we remount with lzo compression and again create a new file. 

In [15]:
sudo mount -o remount,compress=lzo /mnt/btrfs

+ sudo mount -o remount,compress=lzo /mnt/btrfs


In [16]:
sudo dd if=/dev/zero of=/mnt/btrfs/testfile2 bs=1M count=100

+ sudo dd if=/dev/zero of=/mnt/btrfs/testfile2 bs=1M count=100
100+0 records in
100+0 records out
104857600 bytes (105 MB, 100 MiB) copied, 0.0657644 s, 1.6 GB/s


### 3. Compare compression
We use **btrfs-compsize** for this. Compsize takes a list of files on a btrfs filesystem (recursing directories) and measures used compression types and the effective compression ratio.

https://man.archlinux.org/man/extra/compsize/compsize.8.en

In [17]:
sudo apt-get install btrfs-compsize

+ sudo apt-get install btrfs-compsize
Reading package lists... 0%Reading package lists... 100%Reading package lists... Done
Building dependency tree... 0%Building dependency tree... 0%Building dependency tree... 50%Building dependency tree... 50%Building dependency tree       
Reading state information... 0%Reading state information... 0%Reading state information... Done
The following NEW packages will be installed:
  btrfs-compsize
0 upgraded, 1 newly installed, 0 to remove and 83 not upgraded.
Need to get 11.4 kB of archives.
After this operation, 38.9 kB of additional disk space will be used.
Get:1 http://nova.clouds.archive.ubuntu.com/ubuntu focal/universe amd64 btrfs-compsize amd64 1.3-2 [11.4 kB]
Fetched 11.4 kB in 0s (46.2 kB/s)   
debconf: unable to initialize frontend: Dialog
debconf: (Dialog frontend will not work on a dumb terminal, an emacs shell buffer, or without a controlling terminal.)
debconf: falling back to frontend: Readline
Selecting previously unselected package b

In [149]:
sudo compsize /mnt/btrfs/

+ sudo compsize /mnt/btrfs/
Processed 2 files, 802 regular extents (802 refs), 0 inline.
Type       Perc     Disk Usage   Uncompressed Referenced  
TOTAL       51%      103M         200M         200M       
none       100%      100M         100M         100M       
lzo          3%      3.1M         100M         100M       


From this we can understand that compression is working effectively.
* LZO compression reduced 100MB of data to 3.1MB (97% savings).
* Overall storage use was reduced from 200MB to 103MB (about 51% of the original).

We again disable compression by remounting to carry out rest of operations.

In [18]:
sudo mount -o remount,compress=no /mnt/btrfs

+ sudo mount -o remount,compress=no /mnt/btrfs


## Subvolumes and Snapshots
A subvolume in Btrfs is a unique feature that allows you to create independent storage units within the same Btrfs filesystem. Unlike traditional directories, subvolumes can be snapshotted, backed up, and managed separately while sharing the same storage space. This makes them highly useful for system organization, backups, and isolating data.

https://www.man7.org/linux/man-pages/man8/btrfs-subvolume.8.html

### 1. Creating a Subvolume
To create a subvolume, use the following command:

In [19]:
sudo btrfs subvolume create /mnt/btrfs/my_subvolume

+ sudo btrfs subvolume create /mnt/btrfs/my_subvolume
Create subvolume '/mnt/btrfs/my_subvolume'


**btrfs subvolume create:** This command creates a new subvolume.

**/mnt/btrfs/my_subvolume:** The path where the subvolume will be created.

After creation, the subvolume will act like a directory but has special properties that allow snapshots and independent storage control.

### 2. Listing Subvolumes
To see all subvolumes within your mounted Btrfs partition, run:

In [20]:
sudo btrfs subvolume list /mnt/btrfs

+ sudo btrfs subvolume list /mnt/btrfs
ID 256 gen 9 top level 5 path my_subvolume


**btrfs subvolume list:** Lists all subvolumes inside the given Btrfs mount point.

This output shows that a subvolume with ID 256 exists at /mnt/btrfs/my_subvolume. A directory representing a subvolume has always inode number 256 (sometimes also called a root of the subvolume).

### 3. Accessing Subvolume
A subvolume in BTRFS can be accessed in two ways:
* like any other directory that is accessible to the user
* like a separately mounted filesystem (options subvol or subvolid)

#### (a) Accessing the Subvolume as a Normal Directory
If the entire Btrfs filesystem is mounted at /mnt/btrfs, you can simply navigate to the subvolume like a regular directory:

In [21]:
cd /mnt/btrfs/my_subvolume
ls

+ cd /mnt/btrfs/my_subvolume
+ ls --color=auto


#### (b)  Accessing the Subvolume via Separate Mounting
For this you first have to create a mount directory using **mkdir**

In [22]:
sudo mkdir -p /mnt/my_subvolume

+ sudo mkdir -p /mnt/my_subvolume


Now, you can mount the subvolume using this command:

In [23]:
sudo mount -o subvol=my_subvolume /dev/vdb1 /mnt/my_subvolume

+ sudo mount -o subvol=my_subvolume /dev/vdb1 /mnt/my_subvolume


Subvolumes can be mounted like file system partitions using the **subvol=/path/to/subvolume** or **subvolid=objectid** mount flags. Also, mounting a subvolume separately does not copy its contents to another folder. Instead, it provides an independent view of that subvolume, making it accessible as if it were a standalone filesystem. This can be useful for isolating data, backups, or creating independent mount points.

https://man.archlinux.org/man/btrfs.5#MOUNT_OPTIONS

#### Key Differences Between Accessing as a Directory vs. Mounting as a Subvolume

| Method | How You Access It | Behavior |
|:--------:|:--------:|:--------:|
|  Accessing as a directory **(/mnt/btrfs/my_subvolume)**   |  You navigate into it like a regular directory inside the mounted Btrfs partition **(/mnt/btrfs)**.   |  The subvolume exists within the main Btrfs filesystem, and you see it as part of the directory structure.   |
|  Mounting separately **(/mnt/my_subvolume)**   |  You mount it explicitly using **mount -o subvol=my_subvolume**.   |  The subvolume behaves like an independent filesystem, isolated from the main Btrfs mount point.   |

### 4. Taking Snapshots of a Subvolume
A snapshot in Btrfs is a point-in-time copy of a subvolume. Unlike traditional copies, Btrfs snapshots are space-efficient because they use copy-on-write (CoW). This means:

* No data is duplicated initially—the snapshot just references existing data.
* Changes are only stored when modifications occur, minimizing disk space usage.

#### (a) Creating a writable snapshot
A read-write (RW) snapshot allows modifications. It can be created using the command:

In [24]:
sudo btrfs subvolume snapshot /mnt/btrfs/my_subvolume /mnt/btrfs/my_snapshot

+ sudo btrfs subvolume snapshot /mnt/btrfs/my_subvolume /mnt/btrfs/my_snapshot
Create a snapshot of '/mnt/btrfs/my_subvolume' in '/mnt/btrfs/my_snapshot'


**btrfs subvolume snapshot**: Creates a snapshot of an existing subvolume.

**/mnt/btrfs/my_subvolume**: The subvolume being snapshotted.

**/mnt/btrfs/my_snapshot**: The snapshot destination (a new subvolume).

This snapshot acts like a normal subvolume only, i.e you can modify files within it.

**(b) Creating a read only snapshot**

This can be created for backup purposes by adding the **-r** flag.

In [25]:
sudo btrfs subvolume snapshot -r /mnt/btrfs/my_subvolume /mnt/btrfs/my_snapshot_r

+ sudo btrfs subvolume snapshot -r /mnt/btrfs/my_subvolume /mnt/btrfs/my_snapshot_r
Create a readonly snapshot of '/mnt/btrfs/my_subvolume' in '/mnt/btrfs/my_snapshot_r'


### 5. Rolling back to the snapshot
We can delete the original subvolume and then restore it by essentially creating a new snapshot out of the existing snapshot

In [26]:
sudo btrfs subvolume delete /mnt/btrfs/my_subvolume

+ sudo btrfs subvolume delete /mnt/btrfs/my_subvolume
Delete subvolume (no-commit): '/mnt/btrfs/my_subvolume'


In [27]:
sudo btrfs subvolume snapshot /mnt/btrfs/my_snapshot /mnt/btrfs/my_subvolume

+ sudo btrfs subvolume snapshot /mnt/btrfs/my_snapshot /mnt/btrfs/my_subvolume
Create a snapshot of '/mnt/btrfs/my_snapshot' in '/mnt/btrfs/my_subvolume'


### 6. Deleting a snapshot
We can use **subvolume delete** command for deleting a snapshot since a snapshot is a subvolume by itself. Deleting a snapshot does not affect the original subvolume. 

https://man7.org/linux/man-pages/man8/btrfs-subvolume.8.html

In [28]:
sudo btrfs subvolume delete /mnt/btrfs/my_snapshot
sudo btrfs subvolume delete /mnt/btrfs/my_snapshot_r

+ sudo btrfs subvolume delete /mnt/btrfs/my_snapshot
Delete subvolume (no-commit): '/mnt/btrfs/my_snapshot'
+ sudo btrfs subvolume delete /mnt/btrfs/my_snapshot_r
Delete subvolume (no-commit): '/mnt/btrfs/my_snapshot_r'


### 7. Deleting Subvolume
If you need to delete a subvolume, we use the command:

In [29]:
sudo btrfs subvolume delete /mnt/btrfs/my_subvolume

+ sudo btrfs subvolume delete /mnt/btrfs/my_subvolume
Delete subvolume (no-commit): '/mnt/btrfs/my_subvolume'


**btrfs subvolume delete**: Deletes the specified subvolume.

**/mnt/btrfs/my_subvolume**: The path of the subvolume to delete.

⚠️ <font color='red'>If the subvolume contains files, they will also be deleted. Ensure that you have backed up important data before removing a subvolume</font>

## Copy-on-Write (CoW) Behavior
Btrfs uses Copy-on-Write (CoW) to prevent data corruption by ensuring that modified data is written to a new location before updating metadata. This means that data blocks are not overwritten directly.
### Testing CoW with File Copy
You can test CoW by creating a reflink copy, where the new file shares the same blocks as the original until modified: 

In [31]:
sudo cp --reflink=always /mnt/btrfs/testfile /mnt/btrfs/testfile3

+ sudo cp --reflink=always /mnt/btrfs/testfile /mnt/btrfs/testfile3


Reflink is a type of shallow copy of file data that shares the blocks but otherwise the files are independent and any change to the file will not affect the other. This builds on the underlying COW mechanism. A reflink will effectively create only a separate metadata pointing to the shared blocks which is typically much faster than a deep copy of all blocks.

https://btrfs.readthedocs.io/en/latest/Reflink.html

Now, we will use the **filefrag** tool which will give us insights into the file fragmentation and layout which we can then use to know the differences before and after file modification. 

In [32]:
sudo filefrag -v /mnt/btrfs/testfile
sudo filefrag -v /mnt/btrfs/testfile3

+ sudo filefrag -v /mnt/btrfs/testfile
Filesystem type is: 9123683e
File size of /mnt/btrfs/testfile is 104857600 (25600 blocks of 4096 bytes)
 ext:     logical_offset:        physical_offset: length:   expected: flags:
   0:        0..   12799:      15616..     28415:  12800:             shared
   1:    12800..   25599:      32000..     44799:  12800:      28416: last,shared,eof
/mnt/btrfs/testfile: 2 extents found
+ sudo filefrag -v /mnt/btrfs/testfile3
Filesystem type is: 9123683e
File size of /mnt/btrfs/testfile3 is 104857600 (25600 blocks of 4096 bytes)
 ext:     logical_offset:        physical_offset: length:   expected: flags:
   0:        0..   12799:      15616..     28415:  12800:             shared
   1:    12800..   25599:      32000..     44799:  12800:      28416: last,shared,eof
/mnt/btrfs/testfile3: 2 extents found


In [33]:
echo 'This is a changed test file' | sudo tee /mnt/btrfs/testfile3 > /dev/null

+ sudo tee /mnt/btrfs/testfile3
+ echo 'This is a changed test file'


In [34]:
sudo filefrag -v /mnt/btrfs/testfile
sudo filefrag -v /mnt/btrfs/testfile3

+ sudo filefrag -v /mnt/btrfs/testfile
Filesystem type is: 9123683e
File size of /mnt/btrfs/testfile is 104857600 (25600 blocks of 4096 bytes)
 ext:     logical_offset:        physical_offset: length:   expected: flags:
   0:        0..   12799:      15616..     28415:  12800:            
   1:    12800..   25599:      32000..     44799:  12800:      28416: last,eof
/mnt/btrfs/testfile: 2 extents found
+ sudo filefrag -v /mnt/btrfs/testfile3
Filesystem type is: 9123683e
File size of /mnt/btrfs/testfile3 is 28 (1 block of 4096 bytes)
 ext:     logical_offset:        physical_offset: length:   expected: flags:
   0:        0..    4095:          0..      4095:   4096:             last,not_aligned,inline,eof
/mnt/btrfs/testfile3: 1 extent found


The filefrag tool gives insight into file fragmentation and layout. Here's a brief overview of the output fields:
* ext: The extent number for the file.
* logical_offset: The logical offset (range) of the file blocks.
* physical_offset: The physical location on disk where the data resides.
* length: The length of the extent in blocks.
* flags:
  * shared: Indicates that the extent is shared by multiple files.
  * last: Marks the final extent of the file.
  * inline: Denotes inline storage for small files.

https://man7.org/linux/man-pages/man8/filefrag.8.html

Initially, both testfile and testfile3 shared the same data blocks, as evidenced by the "shared" flag in the filefrag output. This indicates that CoW was functioning, where both files initially used the same physical storage blocks.

After modifying testfile3, the "shared" flag disappeared, meaning that the file was modified and now has its own separate data blocks.

## Deduplication
Deduplication in Btrfs helps reduce the amount of space used on disk by ensuring that identical blocks of data are stored only once. This can significantly save storage space, especially in scenarios where multiple files contain identical data, such as with backups or large datasets. It’s a process of looking up identical data blocks tracked separately and creating a shared logical link while removing one of the copies of the data blocks. This leads to data space savings while it increases metadata consumption.

There are two main deduplication types:
* in-band: Deduplication occurs while the data is being written to disk.
* out-of-band: Deduplication occurs after the data is written and stored.

In the context of Btrfs, in-band deduplication isn't built-in, but tools like **duperemove** provide out-of-band deduplication, where the system goes through data after the fact to remove duplicates.

https://btrfs.readthedocs.io/en/latest/Deduplication.html

### Using duperemove for deduplication
Duperemove is a simple tool for finding duplicated extents and submitting them for deduplication. When given a list of files it will hash their contents on an extent by extent basis and compare those hashes to each other, finding and categorizing extents that match each other.
#### 1. Install duperemove
https://github.com/markfasheh/duperemove

In [35]:
sudo apt install duperemove

+ sudo apt install duperemove
Reading package lists... 0%Reading package lists... 100%Reading package lists... Done
Building dependency tree... 0%Building dependency tree... 0%Building dependency tree... 50%Building dependency tree... 50%Building dependency tree       
Reading state information... 0%Reading state information... 0%Reading state information... Done
sh: 0: getcwd() failed: No such file or directory
The following NEW packages will be installed:
  duperemove
sh: 0: getcwd() failed: No such file or directory
0 upgraded, 1 newly installed, 0 to remove and 83 not upgraded.
sh: 0: getcwd() failed: No such file or directory
Need to get 70.6 kB of archives.
After this operation, 260 kB of additional disk space will be used.
Get:1 http://nova.clouds.archive.ubuntu.com/ubuntu focal/universe amd64 duperemove amd64 0.11.1-3 [70.6 kB]
Fetched 70.6 kB in 0s (250 kB/s)
sh: 0: getcwd() failed: No such file or directory
sh: 0: getcwd() failed: No such file or directory
sh: 0: getcwd() fai

#### 2.  Scanning for Duplicate Blocks
You can scan a directory or mount point (e.g., /mnt/btrfs) for duplicate blocks using:

In [36]:
duperemove -r /mnt/btrfs

+ duperemove -r /mnt/btrfs
Gathering file list...
Using 1 threads for file hashing phase
[1/2] (50.00%) csum: /mnt/btrfs/testfile
[2/2] (100.00%) csum: /mnt/btrfs/testfile2
Total files:  2
Total hashes: 1600
Loading only duplicated hashes from hashfile.
Hashing completed. Using 1 threads to calculate duplicate extents. This may take some time.
[########################################]
Search completed with no errors.             
Simple read and compare of file data found 1 instances of extents that might benefit from deduplication.
Showing 2 identical extents of length 104857600 with id 19b81479
Start		Filename
0	"/mnt/btrfs/testfile"
0	"/mnt/btrfs/testfile2"


This will recursively scan all the files in the specified directory and check for duplicate blocks. It doesn't remove duplicates yet; it just identifies them.
#### 3. Deduplicating Duplicate Blocks:
To actually remove duplicate blocks and reclaim disk space, you need to run the deduplication operation using **duperemove -dr**:
* The -d flag indicates that deduplication should be performed.
* The -r flag ensures that the operation is performed recursively on the specified directory.

First we create another test file and then run filefrag before and after doing deduplication to find that the **shared** flag will be set after performing deduplication. 

In [37]:
sudo dd if=/dev/zero of=/mnt/btrfs/testfile4 bs=1M count=100

+ sudo dd if=/dev/zero of=/mnt/btrfs/testfile4 bs=1M count=100
100+0 records in
100+0 records out
104857600 bytes (105 MB, 100 MiB) copied, 0.0596281 s, 1.8 GB/s


In [38]:
duperemove -r /mnt/btrfs

+ duperemove -r /mnt/btrfs
Gathering file list...
Using 1 threads for file hashing phase
[1/3] (33.33%) csum: /mnt/btrfs/testfile
[2/3] (66.67%) csum: /mnt/btrfs/testfile2
[3/3] (100.00%) csum: /mnt/btrfs/testfile4
Total files:  3
Total hashes: 2400
Loading only duplicated hashes from hashfile.
Hashing completed. Using 1 threads to calculate duplicate extents. This may take some time.
[########################################]
Search completed with no errors.             
Simple read and compare of file data found 1 instances of extents that might benefit from deduplication.
Showing 3 identical extents of length 104857600 with id 19b81479
Start		Filename
0	"/mnt/btrfs/testfile"
0	"/mnt/btrfs/testfile4"
0	"/mnt/btrfs/testfile2"


In [39]:
sudo filefrag -v /mnt/btrfs/testfile
sudo filefrag -v /mnt/btrfs/testfile4

+ sudo filefrag -v /mnt/btrfs/testfile
Filesystem type is: 9123683e
File size of /mnt/btrfs/testfile is 104857600 (25600 blocks of 4096 bytes)
 ext:     logical_offset:        physical_offset: length:   expected: flags:
   0:        0..   12799:      15616..     28415:  12800:            
   1:    12800..   25599:      32000..     44799:  12800:      28416: last,eof
/mnt/btrfs/testfile: 2 extents found
+ sudo filefrag -v /mnt/btrfs/testfile4
Filesystem type is: 9123683e
File size of /mnt/btrfs/testfile4 is 104857600 (25600 blocks of 4096 bytes)
 ext:     logical_offset:        physical_offset: length:   expected: flags:
   0:        0..   12799:      48384..     61183:  12800:            
   1:    12800..   25599:      64768..     77567:  12800:      61184: last,eof
/mnt/btrfs/testfile4: 2 extents found


In [40]:
sudo duperemove -dr /mnt/btrfs

+ sudo duperemove -dr /mnt/btrfs
Gathering file list...
Using 1 threads for file hashing phase
[1/3] (33.33%) csum: /mnt/btrfs/testfile
[2/3] (66.67%) csum: /mnt/btrfs/testfile2
[3/3] (100.00%) csum: /mnt/btrfs/testfile4
Total files:  3
Total hashes: 2400
Loading only duplicated hashes from hashfile.
Hashing completed. Using 1 threads to calculate duplicate extents. This may take some time.
[########################################]
Search completed with no errors.             
Simple read and compare of file data found 1 instances of extents that might benefit from deduplication.
Showing 3 identical extents of length 104857600 with id 19b81479
Start		Filename
0	"/mnt/btrfs/testfile"
0	"/mnt/btrfs/testfile4"
0	"/mnt/btrfs/testfile2"
Using 1 threads for dedupe phase
[0x565088d0c0c0] (1/1) Try to dedupe extents with id 19b81479
[0x565088d0c0c0] Dedupe 2 extents (id: 19b81479) with target: (0, 104857600), "/mnt/btrfs/testfile"
Comparison of extent info shows a net change in shared extents

In [41]:
sudo filefrag -v /mnt/btrfs/testfile
sudo filefrag -v /mnt/btrfs/testfile4

+ sudo filefrag -v /mnt/btrfs/testfile
Filesystem type is: 9123683e
File size of /mnt/btrfs/testfile is 104857600 (25600 blocks of 4096 bytes)
 ext:     logical_offset:        physical_offset: length:   expected: flags:
   0:        0..   12799:      15616..     28415:  12800:             shared
   1:    12800..   25599:      32000..     44799:  12800:      28416: last,shared,eof
/mnt/btrfs/testfile: 2 extents found
+ sudo filefrag -v /mnt/btrfs/testfile4
Filesystem type is: 9123683e
File size of /mnt/btrfs/testfile4 is 104857600 (25600 blocks of 4096 bytes)
 ext:     logical_offset:        physical_offset: length:   expected: flags:
   0:        0..   12799:      15616..     28415:  12800:             shared
   1:    12800..   25599:      32000..     44799:  12800:      28416: last,shared,eof
/mnt/btrfs/testfile4: 2 extents found


## RAID
Btrfs provides built-in RAID support at the file system level, unlike traditional RAID setups that rely on mdadm or hardware RAID controllers. This means that Btrfs can handle redundancy, mirroring, and striping directly within the file system itself.
### Creating a RAID 1 (Mirroring) Setup
RAID 1 ensures that data is duplicated across two or more disks, so if one disk fails, the other still has a complete copy.

https://btrfs.readthedocs.io/en/latest/Volume-management.html
#### 1. Add the Second Disk (/dev/vdc1)
First we check the existing btrfs file system:

In [42]:
sudo btrfs filesystem show

+ sudo btrfs filesystem show
Label: 'mybtrfs'  uuid: 1fca2654-5e68-48b1-85e0-260bafb99ce7
	Total devices 1 FS bytes used 100.27MiB
	devid    1 size 498.00MiB used 272.00MiB path /dev/vdb1



Then we create partition on the disk /dev/vdc

In [44]:
sudo sgdisk -n 1:2MiB:500MiB /dev/vdc

+ sudo sgdisk -n 1:2MiB:500MiB /dev/vdc
Creating new GPT entries in memory.
The operation has completed successfully.


In [46]:
sudo fdisk -l /dev/vdc

+ sudo fdisk -l /dev/vdc
Disk /dev/vdc: 1 GiB, 1073741824 bytes, 2097152 sectors
Units: sectors of 1 * 512 = 512 bytes
Sector size (logical/physical): 512 bytes / 512 bytes
I/O size (minimum/optimal): 512 bytes / 512 bytes
Disklabel type: gpt
Disk identifier: 39851B3C-6B07-4091-B285-74D215EBA5D2

Device     Start     End Sectors  Size Type
/dev/vdc1   4096 1024000 1019905  498M Linux filesystem


To add /dev/vdc1 to the Btrfs volume we execute the following command:

https://man7.org/linux/man-pages/man8/btrfs-device.8.html

In [47]:
sudo btrfs device add /dev/vdc1 /mnt/btrfs

+ sudo btrfs device add /dev/vdc1 /mnt/btrfs


#### 2. Convert the File System to RAID 1
Now, we will rebalance the file system to actually distribute the data in RAID 1 mode:

https://man7.org/linux/man-pages/man8/btrfs-balance.8.html

In [48]:
sudo btrfs balance start -dconvert=raid1 -mconvert=raid1 /mnt/btrfs

+ sudo btrfs balance start -dconvert=raid1 -mconvert=raid1 /mnt/btrfs
Done, had to relocate 5 out of 5 chunks


* -dconvert=raid1 → Converts data to RAID 1 (mirrored across both disks).
* -mconvert=raid1 → Converts metadata to RAID 1.

#### 3. Verify RAID 1 Configuration
We can verify the raid status by the df and show commands.

https://man7.org/linux/man-pages/man1/df.1.html

In [50]:
sudo btrfs filesystem df /mnt/btrfs
sudo btrfs filesystem show /mnt/btrfs

+ sudo btrfs filesystem df /mnt/btrfs
Data, RAID1: total=269.00MiB, used=100.06MiB
System, RAID1: total=32.00MiB, used=16.00KiB
Metadata, RAID1: total=64.00MiB, used=240.00KiB
GlobalReserve, single: total=3.25MiB, used=0.00B
+ sudo btrfs filesystem show /mnt/btrfs
Label: 'mybtrfs'  uuid: 1fca2654-5e68-48b1-85e0-260bafb99ce7
	Total devices 2 FS bytes used 100.31MiB
	devid    1 size 498.00MiB used 365.00MiB path /dev/vdb1
	devid    2 size 498.00MiB used 365.00MiB path /dev/vdc1



This confirms that both data and metadata are mirrored across /dev/vdb1 and /dev/vdc1. Now even if one of the disks fails, the file system would still function because of mirroring. if we remove a particular disk we will be able ro re-add the disk by **device add** and then rebalance the filesystem by **btrfs balance**.

## Tearing down and Clean up
### 1. Unmounting the file systems on /dev/vdb1
Unmounting a file/folder means it is inaccessible for the device to read and make any modification, therefore before overwriting the data to make the recovery harder we need to unmount the file system on /dev/vdb1. We can unmount the chosen partition using the command "sudo umount /dev/partition_name".

We must change directory out of the mounted device first, or else umount will fail and warn that the device is busy.

https://man7.org/linux/man-pages/man8/umount.8.html

In [52]:
cd /home/cc

sudo umount /dev/vdb1

sudo umount /mnt/my_subvolume

+ cd /home/cc
+ sudo umount /dev/vdb1
umount: /dev/vdb1: not mounted.
+ sudo umount /mnt/my_subvolume
umount: /mnt/my_subvolume: not mounted.


: 32

### 2. Checking the drive partition /dev/vdb1 is correctly unmounted
You can use the “lsblk” command and specify the device name.

lsblk lists information about all available or specified block devices. The lsblk command reads the sysfs filesystem and udev db to gather information. If the udev db is not available or lsblk is compiled without udev support, then it tries to read. LABELs, UUIDs, and filesystem types from the block device. In this case, root permissions are necessary.

https://man7.org/linux/man-pages/man8/lsblk.8.html

To check that the partition is properly unmounted, the MOUNTPOINT field should be empty in the output.

In [53]:
lsblk /dev/vdb1

+ lsblk /dev/vdb1
NAME MAJ:MIN RM  SIZE RO TYPE MOUNTPOINT
vdb1 252:17   0  498M  0 part 


### 3. Overwriting data to make recovery harder
We can overwrite the data of a chosen file or devi using the shred command. The shred command is a useful security tool that overwrites a file several times to make it harder to recover its data. We use the "-n" argument to indicate we want to overwrite 2 times, the "-v" argument to provide verbose output, and the "-z" argument to add a final zero to hide the shredding process.

https://man7.org/linux/man-pages/man1/shred.1.html

In [54]:
sudo shred -vz -n 2 /dev/vdb1

+ sudo shred -vz -n 2 /dev/vdb1
shred: /dev/vdb1: pass 1/3 (random)...
shred: /dev/vdb1: pass 2/3 (random)...
shred: /dev/vdb1: pass 3/3 (000000)...


### 4. Deleting the partition
After unformatting we can finally delete the partition using the command "sudo parted /dev/vdb rm 1"

GNU Parted manipulates partition tables. This is useful for creating space for new operating systems, reorganizing disk usage, copying data on hard disks, and disk imaging.

https://man7.org/linux/man-pages/man8/parted.8.html

In [55]:
sudo parted /dev/vdb rm 1

sudo fdisk -l /dev/vdb

+ sudo parted /dev/vdb rm 1
Information: You may need to update /etc/fstab.

+ sudo fdisk -l /dev/vdb                                                  
Disk /dev/vdb: 1 GiB, 1073741824 bytes, 2097152 sectors
Units: sectors of 1 * 512 = 512 bytes
Sector size (logical/physical): 512 bytes / 512 bytes
I/O size (minimum/optimal): 512 bytes / 512 bytes
Disklabel type: gpt
Disk identifier: 56BC065D-4AB0-45E1-AFF8-CA061BD3A532


## Conclusion
Btrfs is a powerful and feature-rich filesystem that is well-suited for modern storage needs. Its native support for snapshots, subvolumes, deduplication, and RAID makes it a great choice for system administrators and power users who need advanced storage capabilities.


⚠️ <font color='red'>After you are finished with this demo, it is important to be mindful of your resource usage and to “free” resources when you are finished with them, as Chameleon is a shared facility. Removing the resources will revoke your access to them, and all the information stored on them will be erased. Therefore, ensure that you have saved all of your work before deleting the resources. Release floating IPs and delete the instances right after deleting the volumes. </font>